# ClickHouse Alchemy Demo

This notebook demonstrates how to use clickhouse-alchemy to build ClickHouse SQL statements.

In [ ]:
from clickhouse_alchemy import (
    # Schema
    Table, Column, MetaData,
    # Types
    UInt64, UInt32, String, DateTime, Float64,
    Nullable, Array, LowCardinality, Decimal,
    # Engines
    MergeTree, ReplacingMergeTree,
    # SQL
    select, insert, create_table, alter_table,
    column, literal, and_, or_, func,
)

## 1. Define Tables

In [ ]:
metadata = MetaData()

users = Table(
    "users",
    Column("id", UInt64),
    Column("name", String),
    Column("email", Nullable(String)),
    Column("status", LowCardinality(String)),
    Column("created_at", DateTime),
    engine=MergeTree(order_by="id"),
    metadata=metadata,
)

orders = Table(
    "orders",
    Column("id", UInt64),
    Column("user_id", UInt64),
    Column("amount", Decimal(18, 2)),
    Column("tags", Array(String)),
    Column("created_at", DateTime),
    engine=MergeTree(order_by="id"),
    metadata=metadata,
)

print("Tables defined:", list(metadata.tables.keys()))

## 2. CREATE TABLE

In [ ]:
stmt = (
    create_table(users)
    .if_not_exists()
    .order_by("id")
    .partition_by("toYYYYMM(created_at)")
)

print(stmt.compile())

## 3. Basic SELECT Queries

In [ ]:
# Simple select
stmt = select(users.c.id, users.c.name).select_from(users)
print("Simple SELECT:")
print(stmt.compile())
print()

# With WHERE clause
stmt = (
    select(users.c.id, users.c.name)
    .select_from(users)
    .where(users.c.status == "active")
    .where(users.c.created_at > literal("2024-01-01"))
)
print("With WHERE:")
print(stmt.compile())
print()

# ORDER BY, LIMIT
stmt = (
    select(users.c.id, users.c.name)
    .select_from(users)
    .order_by(users.c.created_at.desc())
    .limit(10)
)
print("With ORDER BY and LIMIT:")
print(stmt.compile())

## 4. JOINs and Aggregations

In [ ]:
# JOIN with aggregation
stmt = (
    select(
        users.c.name,
        func.count().label("order_count"),
        func.sum(orders.c.amount).label("total_spent"),
    )
    .select_from(users)
    .left_join(orders, orders.c.user_id == users.c.id)
    .group_by(users.c.name)
    .having(func.count() > 5)
    .order_by(func.sum(orders.c.amount).desc())
    .limit(10)
)

print(stmt.compile())

## 5. ClickHouse-Specific Features

In [ ]:
# FINAL - for deduplicated reads from ReplacingMergeTree
stmt = (
    select(users.c.id, users.c.name)
    .select_from(users)
    .final()
)
print("FINAL:")
print(stmt.compile())
print()

# PREWHERE - early filtering before reading columns
stmt = (
    select(users.c.id, users.c.name, users.c.email)
    .select_from(users)
    .prewhere(users.c.created_at > literal("2024-01-01"))
    .where(users.c.status == "active")
)
print("PREWHERE:")
print(stmt.compile())
print()

# SAMPLE - approximate queries
stmt = (
    select(func.count(), func.avg(orders.c.amount))
    .select_from(orders)
    .sample(0.1)  # 10% sample
)
print("SAMPLE:")
print(stmt.compile())
print()

# ARRAY JOIN - expand arrays into rows
stmt = (
    select(orders.c.id, column("tag"))
    .select_from(orders)
    .array_join(orders.c.tags)
)
print("ARRAY JOIN:")
print(stmt.compile())
print()

# Query SETTINGS
stmt = (
    select(users.c.id)
    .select_from(users)
    .settings(max_threads=4, max_memory_usage=10000000000)
)
print("SETTINGS:")
print(stmt.compile())

## 6. Subqueries and CTEs

In [ ]:
# Subquery in WHERE
high_value_users = (
    select(orders.c.user_id)
    .select_from(orders)
    .group_by(orders.c.user_id)
    .having(func.sum(orders.c.amount) > 10000)
)

stmt = (
    select(users.c.id, users.c.name, users.c.email)
    .select_from(users)
    .where(users.c.id.in_(high_value_users))
)
print("Subquery:")
print(stmt.compile())
print()

# CTE (Common Table Expression)
active_users_cte = (
    select(users.c.id, users.c.name)
    .select_from(users)
    .where(users.c.status == "active")
)

stmt = (
    select(column("id"), column("name"))
    .with_cte("active_users", active_users_cte)
    .select_from(column("active_users"))
)
print("CTE:")
print(stmt.compile())

## 7. INSERT Statements

In [ ]:
# Insert with values
stmt = insert(users).values(
    {"id": 1, "name": "Alice", "status": "active"},
    {"id": 2, "name": "Bob", "status": "pending"},
)
print("INSERT:")
print(stmt.compile())
print()

# Insert from SELECT
stmt = insert(users).from_select(
    ["id", "name", "status"],
    select(users.c.id, users.c.name, literal("archived"))
    .select_from(users)
    .where(users.c.status == "inactive")
)
print("INSERT FROM SELECT:")
print(stmt.compile())

## 8. ALTER TABLE (Mutations)

In [ ]:
# DELETE mutation (ClickHouse's alternative to DELETE)
stmt = alter_table(users).delete(users.c.status == "deleted")
print("DELETE mutation:")
print(stmt.compile())
print()

# UPDATE mutation
stmt = alter_table(users).update(
    {"status": "inactive"},
    users.c.created_at < literal("2023-01-01")
)
print("UPDATE mutation:")
print(stmt.compile())
print()

# Add column
stmt = alter_table(users).add_column("verified", UInt64(), default=0)
print("ADD COLUMN:")
print(stmt.compile())

## 9. SQL Functions

In [ ]:
# Date/time functions
stmt = select(
    func.to_date(users.c.created_at).label("date"),
    func.to_year(users.c.created_at).label("year"),
    func.to_month(users.c.created_at).label("month"),
    func.to_start_of_month(users.c.created_at).label("month_start"),
).select_from(users).limit(5)
print("Date functions:")
print(stmt.compile())
print()

# String functions
stmt = select(
    users.c.name,
    func.length(users.c.name).label("len"),
    func.upper(users.c.name).label("upper_name"),
).select_from(users)
print("String functions:")
print(stmt.compile())
print()

# Conditional functions
stmt = select(
    users.c.id,
    func.if_(
        users.c.status == "active",
        "Active User",
        "Inactive User"
    ).label("status_label"),
).select_from(users)
print("Conditional:")
print(stmt.compile())

## 10. Connecting to ClickHouse

To actually execute queries, create an engine and connection:

In [ ]:
# Example connection (requires a running ClickHouse server)
from clickhouse_alchemy import create_engine

# Create engine (uncomment to use)
# engine = create_engine("clickhouse://localhost:8123/default")

# Execute queries
# with engine.connect() as conn:
#     # Create table
#     conn.execute(create_table(users).if_not_exists())
#
#     # Bulk insert (efficient)
#     conn.insert(users, [
#         {"id": 1, "name": "Alice", "status": "active"},
#         {"id": 2, "name": "Bob", "status": "active"},
#     ])
#
#     # Query
#     result = conn.execute(
#         select(users.c.id, users.c.name).limit(10)
#     )
#     for row in result:
#         print(row)

print("Connection example (commented out - requires ClickHouse server)")